# импорт БД, создание трейн/тест выборок

Заметки на будущщее  

При этом я хочу сделать окна длины 1сек. Смысл в том, что окна ДЭПД центрированы по, собственно, ДЭПД. Это неприемлемо. ДЭПД максимальную длину будем считать за 0.7сек - то есть 0.45с в обе стороны от центра окна с ДЭПД сохраниться просто обязаны! И все окна с ДЭПД просто трансформируем сдвигами. 

Для каждого окна с ДЭПД создадим 11 окон (со сдвигами 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0). И каждому из них присвоим значение - расстояние центра окна маленького до центра окна большого + 0.2 и нормировка на 1.2. Это нужно, чтобы достаточно четко отделять фон - просто нули, от того, что пересекается с ДЭПД +-. Но это делаем потом.

In [3]:
# %% [markdown]
# # Загрузка и анализ датасета bipolar_2sec_dataset.pkl

# %%
import pickle
import numpy as np
from pathlib import Path

# Путь к файлу (указываем тот же, куда сохранили)
dataset_path = Path(r"..\bipolar_2sec_dataset.pkl")

# Загрузка
with open(dataset_path, 'rb') as f:
    dataset = pickle.load(f)

print(f"Загружено образцов: {len(dataset)}")
print(f"Тип датасета: {type(dataset)}")
print(f"Тип первого элемента: {type(dataset[0])}\n")

# Получаем все ключи (поля), которые есть в словарях
all_keys = set()
for sample in dataset:
    all_keys.update(sample.keys())
all_keys = sorted(all_keys)

print("Список всех полей (колонок) в датасете:")
for i, key in enumerate(all_keys, 1):
    print(f"  {i}. {key}")

# Для каждого поля покажем тип значения и пример
print("\nПример значений для каждого поля (на первом образце):")
first = dataset[0]
for key in all_keys:
    value = first[key]
    if key == 'data':
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"  {key}: {value} (тип {type(value).__name__})")

# Дополнительная статистика по категориальным полям
print("\n=== Статистика по категориальным полям ===")
print(f"Уникальные record_id: {len(set(s['record_id'] for s in dataset))}")
print(f"Уникальные external_id: {len(set(s['external_id'] for s in dataset))}")
print(f"Распределение window_type: DEPD={sum(s['window_type']==1 for s in dataset)}, фон={sum(s['window_type']==0 for s in dataset)}")
print(f"Распределение state: awake={sum(s['state']=='awake' for s in dataset)}, sleep={sum(s['state']=='sleep' for s in dataset)}")

# По длительности окон (они все должны быть 2 секунды, но проверим)
durations = [s['end_sec'] - s['start_sec'] for s in dataset]
print(f"Длительность окон: мин={min(durations):.3f} с, макс={max(durations):.3f} с, среднее={np.mean(durations):.3f} с")

# Вывод информации о форме данных (каналы × отсчёты)
shapes = [s['data'].shape for s in dataset]
unique_shapes = set(shapes)
print(f"Уникальные формы данных: {unique_shapes}")
if len(unique_shapes) == 1:
    n_ch, n_samples = shapes[0]
    print(f"  Все образцы имеют {n_ch} каналов и {n_samples} отсчётов (при {n_samples/500:.1f} с при 500 Гц)")

Загружено образцов: 14131
Тип датасета: <class 'list'>
Тип первого элемента: <class 'dict'>

Список всех полей (колонок) в датасете:
  1. data
  2. end_sec
  3. external_id
  4. original_episode_duration
  5. record_id
  6. sfreq
  7. start_sec
  8. state
  9. window_type

Пример значений для каждого поля (на первом образце):
  data: shape=(18, 1000), dtype=float32
  end_sec: 4.791 (тип float)
  external_id: 000_FA0183FS (тип str)
  original_episode_duration: 0.22999999999999998 (тип float)
  record_id: 2026_05_04_000002 (тип str)
  sfreq: 500.0 (тип float)
  start_sec: 2.7910000000000004 (тип float)
  state: awake (тип str)
  window_type: 1 (тип int)

=== Статистика по категориальным полям ===
Уникальные record_id: 57
Уникальные external_id: 57
Распределение window_type: DEPD=4131, фон=10000
Распределение state: awake=4311, sleep=9820
Длительность окон: мин=2.000 с, макс=2.000 с, среднее=2.000 с
Уникальные формы данных: {(18, 1000)}
  Все образцы имеют 18 каналов и 1000 отсчётов (при 

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from collections import Counter

# Сначала получим список уникальных пациентов
unique_patients = list(set([sample['external_id'] for sample in dataset]))
print(f"Всего пациентов: {len(unique_patients)}")
print(f"Распределение классов по пациентам:")

# Посмотрим, сколько образцов каждого типа у каждого пациента
patient_stats = []
for patient in unique_patients:
    patient_samples = [s for s in dataset if s['external_id'] == patient]
    n_depd = sum(1 for s in patient_samples if s['window_type'] == 1)
    n_background = len(patient_samples) - n_depd
    patient_stats.append({
        'patient': patient,
        'total': len(patient_samples),
        'DEPD': n_depd,
        'background': n_background,
        'ratio': n_depd / len(patient_samples) if len(patient_samples) > 0 else 0
    })

patient_stats_df = pd.DataFrame(patient_stats)
print(patient_stats_df.sort_values('DEPD', ascending=False).head(10))
print(f"\nСреднее отношение DEPD/всего на пациента: {patient_stats_df['ratio'].mean():.3f}")

# Пациент-независимое разбиение (стратифицированное по наличию DEPD)
# Разделим пациентов, а не образцы!
train_patients, test_patients = train_test_split(
    unique_patients, 
    test_size=0.2,  # 20% пациентов - в тест
    random_state=42,
    stratify=[1 if s['DEPD'] > 0 else 0 for s in patient_stats]  # стратифицируем, есть ли у пациента DEPD
)

# Создаём финальные выборки
train_samples = [s for s in dataset if s['external_id'] in train_patients]
test_samples = [s for s in dataset if s['external_id'] in test_patients]

print(f"\n=== Итоги разбиения ===")
print(f"Тренировочные пациенты: {len(train_patients)}")
print(f"Тестовые пациенты: {len(test_patients)}")
print(f"Тренировочных образцов: {len(train_samples)} (DEPD: {sum(1 for s in train_samples if s['window_type']==1)}, фон: {sum(1 for s in train_samples if s['window_type']==0)})")
print(f"Тестовых образцов: {len(test_samples)} (DEPD: {sum(1 for s in test_samples if s['window_type']==1)}, фон: {sum(1 for s in test_samples if s['window_type']==0)})")

Всего пациентов: 57
Распределение классов по пациентам:
                   patient  total  DEPD  background     ratio
37    patient_009_FA0183ZO   2887  1879        1008  0.650849
20            005_FA0183ZO    881   717         164  0.813848
16  patient_1.002_GA1501CO    788   533         255  0.676396
51            000_FA0183FS    408   196         212  0.480392
41            008_GA15011B    842   122         720  0.144893
56          2.048_DA0236PQ     88    46          42  0.522727
46          2.050_DA0236Q8   2605    36        2569  0.013820
50          2.047_FA0015BC     39    35           4  0.897436
42          2.049_GA1500WQ     33    31           2  0.939394
52          2.042_FA0015HC    350    28         322  0.080000

Среднее отношение DEPD/всего на пациента: 0.509

=== Итоги разбиения ===
Тренировочные пациенты: 45
Тестовые пациенты: 12
Тренировочных образцов: 9459 (DEPD: 3019, фон: 6440)
Тестовых образцов: 4672 (DEPD: 1112, фон: 3560)


In [5]:
import json
import pickle
from pathlib import Path

# Создаём директорию для сохранения
SAVE_DIR = Path("./data_splits")
SAVE_DIR.mkdir(exist_ok=True)

def save_train_test_split(train_samples, test_samples, train_patients, test_patients, save_dir=SAVE_DIR):
    """
    Сохраняет train/test разбиение в нескольких форматах для удобства
    """
    # 1. Сохраняем списки пациентов (удобно для просмотра)
    split_info = {
        'train_patients': list(train_patients),
        'test_patients': list(test_patients),
        'statistics': {
            'n_train_patients': len(train_patients),
            'n_test_patients': len(test_patients),
            'n_train_samples': len(train_samples),
            'n_test_samples': len(test_samples),
            'n_train_depd': sum(1 for s in train_samples if s['window_type'] == 1),
            'n_train_background': sum(1 for s in train_samples if s['window_type'] == 0),
            'n_test_depd': sum(1 for s in test_samples if s['window_type'] == 1),
            'n_test_background': sum(1 for s in test_samples if s['window_type'] == 0),
            'patient_distribution': {
                'train_patients_ratio': round(len(train_patients) / (len(train_patients) + len(test_patients)), 3),
                'test_patients_ratio': round(len(test_patients) / (len(train_patients) + len(test_patients)), 3)
            }
        }
    }
    
    # Сохраняем как JSON
    with open(save_dir / 'split_info.json', 'w') as f:
        json.dump(split_info, f, indent=2)
    print(f"✓ Сохранена информация о разбиении в {save_dir / 'split_info.json'}")
    
    # 2. Сохраняем индексы образцов (более надёжный способ)
    # Создаём маппинг внешнего ID к индексу в датасете
    # ВАЖНО: предполагаем, что у вас есть доступ к исходному dataset списку
    # Сохраняем только идентификаторы образцов
    train_ids = [{'record_id': s['record_id'], 
                  'external_id': s['external_id'], 
                  'start_sec': s['start_sec']} for s in train_samples]
    test_ids = [{'record_id': s['record_id'], 
                 'external_id': s['external_id'], 
                 'start_sec': s['start_sec']} for s in test_samples]
    
    with open(save_dir / 'train_sample_ids.pkl', 'wb') as f:
        pickle.dump(train_ids, f)
    with open(save_dir / 'test_sample_ids.pkl', 'wb') as f:
        pickle.dump(test_ids, f)
    print(f"✓ Сохранены идентификаторы тренировочных образцов ({len(train_ids)})")
    print(f"✓ Сохранены идентификаторы тестовых образцов ({len(test_ids)})")
    
    # 3. Сохраняем простые списки пациентов в текстовом формате
    with open(save_dir / 'train_patients.txt', 'w') as f:
        f.write('\n'.join(train_patients))
    with open(save_dir / 'test_patients.txt', 'w') as f:
        f.write('\n'.join(test_patients))
    print(f"✓ Сохранены списки пациентов в текстовом формате")
    
    return split_info

def load_train_test_split(dataset, save_dir=SAVE_DIR):
    """
    Загружает сохранённое разбиение и восстанавливает выборки
    """
    # Загружаем идентификаторы
    with open(save_dir / 'train_sample_ids.pkl', 'rb') as f:
        train_ids = pickle.load(f)
    with open(save_dir / 'test_sample_ids.pkl', 'rb') as f:
        test_ids = pickle.load(f)
    
    # Создаём множество для быстрого поиска
    train_keys = {(item['record_id'], item['start_sec']) for item in train_ids}
    test_keys = {(item['record_id'], item['start_sec']) for item in test_ids}
    
    # Восстанавливаем выборки
    train_samples = []
    test_samples = []
    
    for sample in dataset:
        key = (sample['record_id'], sample['start_sec'])
        if key in train_keys:
            train_samples.append(sample)
        elif key in test_keys:
            test_samples.append(sample)
        else:
            print(f"Warning: Sample {key} not found in saved splits!")
    
    # Загружаем списки пациентов
    with open(save_dir / 'train_patients.txt', 'r') as f:
        train_patients = f.read().splitlines()
    with open(save_dir / 'test_patients.txt', 'r') as f:
        test_patients = f.read().splitlines()
    
    # Загружаем статистику
    with open(save_dir / 'split_info.json', 'r') as f:
        split_info = json.load(f)
    
    print(f"\n=== Загружено разбиение ===")
    print(f"Тренировочные пациенты: {len(train_patients)}")
    print(f"Тестовые пациенты: {len(test_patients)}")
    print(f"Тренировочных образцов: {len(train_samples)} (DEPD: {sum(1 for s in train_samples if s['window_type']==1)}, фон: {sum(1 for s in train_samples if s['window_type']==0)})")
    print(f"Тестовых образцов: {len(test_samples)} (DEPD: {sum(1 for s in test_samples if s['window_type']==1)}, фон: {sum(1 for s in test_samples if s['window_type']==0)})")
    
    return train_samples, test_samples, train_patients, test_patients, split_info

# Пример использования ПОСЛЕ вашего разбиения:

# Предполагаем, что у вас уже есть:
# dataset - полный датасет
# train_patients, test_patients - списки пациентов
# train_samples, test_samples - выборки (которые вы уже создали)

# Сохраняем разбиение
split_info = save_train_test_split(
    train_samples=train_samples,
    test_samples=test_samples,
    train_patients=train_patients,
    test_patients=test_patients
)

print(f"\n=== Статистика сохранённого разбиения ===")
print(json.dumps(split_info['statistics'], indent=2))

✓ Сохранена информация о разбиении в data_splits\split_info.json
✓ Сохранены идентификаторы тренировочных образцов (9459)
✓ Сохранены идентификаторы тестовых образцов (4672)
✓ Сохранены списки пациентов в текстовом формате

=== Статистика сохранённого разбиения ===
{
  "n_train_patients": 45,
  "n_test_patients": 12,
  "n_train_samples": 9459,
  "n_test_samples": 4672,
  "n_train_depd": 3019,
  "n_train_background": 6440,
  "n_test_depd": 1112,
  "n_test_background": 3560,
  "patient_distribution": {
    "train_patients_ratio": 0.789,
    "test_patients_ratio": 0.211
  }
}


In [ ]:
def load_split_with_validation(dataset, split_dir=SAVE_DIR):
    """
    Расширенная версия загрузки с проверкой корректности
    """
    try:
        # Пытаемся загрузить существующее разбиение
        train_samples, test_samples, train_patients, test_patients, split_info = load_train_test_split(dataset, split_dir)
        
        # Проверяем, что пациенты не пересекаются
        overlapping_patients = set(train_patients) & set(test_patients)
        if overlapping_patients:
            print(f"WARNING: Найдены пересекающиеся пациенты: {overlapping_patients}")
        else:
            print("Пересечений пациентов нет")
        
        # Проверяем, что все образцы распределены
        total_loaded = len(train_samples) + len(test_samples)
        if total_loaded == len(dataset):
            print(f"✓ Все {total_loaded} образцов успешно загружены")
        else:
            print(f"WARNING: Загружено {total_loaded} из {len(dataset)} образцов")
        
        return train_samples, test_samples, train_patients, test_patients, split_info
        
    except FileNotFoundError:
        print("Сохраенное разбиение не найдено. Пожалуйста, сначала создайте разбиение с помощью save_train_test_split")
        return None, None, None, None, None


In [7]:
# Пример использования после сохранения:
train_samples, test_samples, train_patients, test_patients, split_info = load_split_with_validation(dataset)


=== Загружено разбиение ===
Тренировочные пациенты: 45
Тестовые пациенты: 12
Тренировочных образцов: 9459 (DEPD: 3019, фон: 6440)
Тестовых образцов: 4672 (DEPD: 1112, фон: 3560)
✓ Пересечений пациентов нет
✓ Все 14131 образцов успешно загружены
